In [1]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import PolynomialFeatures
import pandas as pd
import numpy as np

import mlflow
import mlflow.sklearn
import dagshub
import joblib


In [2]:
mlflow.set_tracking_uri('https://dagshub.com/omalbhare/medical-insurance-cost-prediction-mlops-project.mlflow')
dagshub.init(repo_owner='omalbhare', repo_name='medical-insurance-cost-prediction-mlops-project', mlflow=True)
mlflow.set_experiment("Regression Model Comparison")

Accessing as omalbhare

Initialized MLflow to track repo "omalbhare/medical-insurance-cost-prediction-mlops-project"

Repository omalbhare/medical-insurance-cost-prediction-mlops-project initialized!

<Experiment: artifact_location='mlflow-artifacts:/079e18bf528641898ac7249ab4a9a41a', creation_time=1767182217653, experiment_id='0', last_update_time=1767182217653, lifecycle_stage='active', name='Regression Model Comparison', tags={'mlflow.experimentKind': 'custom_model_development'}>

In [3]:
X_train = joblib.load("X_train.pkl")
X_test  = joblib.load("X_test.pkl")
y_train = joblib.load("y_train.pkl")
y_test  = joblib.load("y_test.pkl")

In [4]:
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))  
    mae  = mean_absolute_error(y_test, y_pred)         
    r2   = r2_score(y_test, y_pred)                     

    return { "test_rmse": rmse, "test_mae": mae, "test_r2": r2    }


In [5]:
results = {}
best_run = { "rmse": float("inf"), "run_id": None, "model_name": None}

for degree in [2, 3]:
    with mlflow.start_run(run_name=f"PolyReg_deg_{degree}"):

        poly = PolynomialFeatures(degree=degree)
        X_train_poly = poly.fit_transform(X_train)
        X_test_poly  = poly.transform(X_test)

        model = LinearRegression()
        model.fit(X_train_poly, y_train)

        metrics = evaluate_model(model, X_test_poly, y_test)

        # Log parameters
        mlflow.log_param("model_name", "PolynomialRegression")
        mlflow.log_param("degree", degree)

        # Log metrics (ONLY test metrics)
        mlflow.log_metrics(metrics)

        # Store locally
        results[f"PolyReg_deg_{degree}"] = { "degree": degree,"metrics": metrics, "run_id": mlflow.active_run().info.run_id }

        # Best model tracking
        if metrics["test_rmse"] < best_run["rmse"]:
            best_run.update({
                "rmse": metrics["test_rmse"],
                "run_id": mlflow.active_run().info.run_id,
                "model_name": f"PolyReg_deg_{degree}"
            })


🏃 View run PolyReg_deg_2 at: https://dagshub.com/omalbhare/medical-insurance-cost-prediction-mlops-project.mlflow/#/experiments/0/runs/e28ff1b1494c4a9599450c7a46b8bbbb
🧪 View experiment at: https://dagshub.com/omalbhare/medical-insurance-cost-prediction-mlops-project.mlflow/#/experiments/0
🏃 View run PolyReg_deg_3 at: https://dagshub.com/omalbhare/medical-insurance-cost-prediction-mlops-project.mlflow/#/experiments/0/runs/111c8c82e07d4caa9bba46f78de3ae15
🧪 View experiment at: https://dagshub.com/omalbhare/medical-insurance-cost-prediction-mlops-project.mlflow/#/experiments/0


In [6]:
models = {
    "LinearRegression": LinearRegression(),
    "RandomForest": RandomForestRegressor(n_estimators=100, random_state=42),
    "GradientBoosting": GradientBoostingRegressor(n_estimators=100, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=100, random_state=42),
    "SVR": SVR()
}

for name, model in models.items():
    with mlflow.start_run(run_name=name):

        # Train
        model.fit(X_train, y_train)

        # Evaluate only on test set
        metrics = evaluate_model(model, X_test, y_test)

        # Log basic params
        mlflow.log_param("model_name", name)

        # Log model hyperparameters safely
        if hasattr(model, "get_params"):
            for param, value in model.get_params().items():
                if isinstance(value, (int, float, str, bool)):
                    mlflow.log_param(param, value)

        # Log only test metrics
        mlflow.log_metrics(metrics)

        # Store results locally
        results[name] = { "metrics": metrics,  "run_id": mlflow.active_run().info.run_id     }

        # Track best model based on test RMSE
        if metrics["test_rmse"] < best_run["rmse"]:
            best_run.update({
                "rmse": metrics["test_rmse"],
                "run_id": mlflow.active_run().info.run_id,
                "model_name": name
            })

print("Best Model:", best_run)

🏃 View run LinearRegression at: https://dagshub.com/omalbhare/medical-insurance-cost-prediction-mlops-project.mlflow/#/experiments/0/runs/4c001938bf23495ea05d1b4e61389522
🧪 View experiment at: https://dagshub.com/omalbhare/medical-insurance-cost-prediction-mlops-project.mlflow/#/experiments/0
🏃 View run RandomForest at: https://dagshub.com/omalbhare/medical-insurance-cost-prediction-mlops-project.mlflow/#/experiments/0/runs/8da1c88aa83d4b62a503e0cc78544dd1
🧪 View experiment at: https://dagshub.com/omalbhare/medical-insurance-cost-prediction-mlops-project.mlflow/#/experiments/0
🏃 View run GradientBoosting at: https://dagshub.com/omalbhare/medical-insurance-cost-prediction-mlops-project.mlflow/#/experiments/0/runs/e73425be255242a7ac7dc8a239de728c
🧪 View experiment at: https://dagshub.com/omalbhare/medical-insurance-cost-prediction-mlops-project.mlflow/#/experiments/0
🏃 View run XGBoost at: https://dagshub.com/omalbhare/medical-insurance-cost-prediction-mlops-project.mlflow/#/experiments/

In [9]:
import pandas as pd

# Convert results dictionary to DataFrame
summary_list = []

for model_name, res in results.items():
    metrics = res["metrics"]
    summary_list.append({
        "model_name": model_name,
        "run_id": res["run_id"],
        "test_rmse": metrics["test_rmse"],
        "test_mae": metrics["test_mae"],
        "test_r2": metrics["test_r2"]
    })

summary_df = pd.DataFrame(summary_list)

# Sort by RMSE ascending (lower RMSE = better)
summary_df = summary_df.sort_values(by="test_rmse").reset_index(drop=True)

summary_df

,model_name,run_id,test_rmse,test_mae,test_r2
0,GradientBoosting,e73425be255242a7ac7dc8a239de728c,4931.692512,3849.462405,0.831256
1,RandomForest,8da1c88aa83d4b62a503e0cc78544dd1,5357.695401,4097.304029,0.800844
2,XGBoost,a4a92d56e05948ffbc3095a17a2e04cf,5470.135090,4174.642964,0.792397
3,PolyReg_deg_2,e28ff1b1494c4a9599450c7a46b8bbbb,5684.827554,4426.321815,0.775781
4,PolyReg_deg_3,111c8c82e07d4caa9bba46f78de3ae15,5754.977281,4357.581275,0.770214
5,LinearRegression,4c001938bf23495ea05d1b4e61389522,6368.137982,5034.263199,0.718640
6,SVR,182a749bc29d456e8823cd6d67160ac3,12601.564227,8170.997899,-0.101757


## hyperparametertunning

In [12]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 4, 5],
    'subsample': [0.8, 1.0]
}

gbr = GradientBoostingRegressor(random_state=42)
grid = GridSearchCV(estimator=gbr, param_grid=param_grid, cv=5, scoring='r2', n_jobs=-1)

# MLflow experiment
with mlflow.start_run(run_name="GradientBoostingRegressor hyperparameter tune") as run:
    # Fit GridSearch
    grid.fit(X_train, y_train)
    
    # Get best model
    best_model = grid.best_estimator_
    
    # Print best hyperparameters
    print("=== Best Hyperparameters ===")
    print(grid.best_params_)
    
    # Log model name and hyperparameters to MLflow
    mlflow.log_param("model_name", "GradientBoostingRegressor")
    for param, value in best_model.get_params().items():
        if isinstance(value, (int, float, str, bool)):
            mlflow.log_param(param, value)
    
    # Evaluate using your function
    metrics = evaluate_model(best_model, X_test, y_test)
    
    # Print evaluation results
    print("\n=== Test Set Evaluation Metrics ===")
    for key, value in metrics.items():
        print(f"{key}: {value:.4f}")
    
    # Log metrics to MLflow
    mlflow.log_metrics(metrics)
    
    print("\nMLflow Run ID:", run.info.run_id)





=== Best Hyperparameters ===
{'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 100, 'subsample': 1.0}

=== Test Set Evaluation Metrics ===
test_rmse: 4931.0661
test_mae: 3845.0102
test_r2: 0.8313

MLflow Run ID: 177ff63522304e75ab315a3a19abb25b
🏃 View run GradientBoostingRegressor hyperparameter tune at: https://dagshub.com/omalbhare/medical-insurance-cost-prediction-mlops-project.mlflow/#/experiments/0/runs/177ff63522304e75ab315a3a19abb25b
🧪 View experiment at: https://dagshub.com/omalbhare/medical-insurance-cost-prediction-mlops-project.mlflow/#/experiments/0
